# Embeddings per generator model, without folds

Each transcript (text with its tags) is encoded as a single vector with `sentence-transformers/bert-base-nli-mean-tokens`. Each level is represented by the mean vector of the draw's examples; every video outside the draw gets the level of the nearest mean (cosine distance by default, as in `embedding_baseline.ipynb`).

Everything is computed **separately for each LLM that generated the videos**: the examples and the evaluated videos always come from the same model.

**No cross-validation split** (a fixed split decides which videos serve as examples, and the results depended on it). For each model, label (human or reference) and number of examples $M$:

1. a **draw** takes $M$ random videos per level (Low, Medium, High) among the videos of the model;
2. every video of the model **not in the draw** is classified with these examples;
3. this is repeated `N_DRAWS` times.

For a given video, the examples are a uniform random set of $M$ videos per level among the *other* videos of its model. Scores are pooled over all (draw, video) pairs: the expected score with $M$ random examples; their 95% CI resamples the draws, and the range of a **single** draw shows how much one choice of examples can change the result.

The draws come from `lib/per_model.py` and depend only on the data, `SEED` and $M$: **the FGW, embedding and LLM notebooks use exactly the same examples** (the LLM uses the first `N_DRAWS` of them), and the same scores, plots and tables. The last section compares the methods already run.

**Sensitivity to $M$**: the same for several values of $M$, to see how many examples are needed. **`M_MAIN`** sets the $M$ of the main results: a number (computed even if it is not in `M_VALUES`), or `"auto"`: for each model, the smallest $M$ that is "enough" according to the sensitivity analysis.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.realpath("../lib"))
from evaluation import LEVELS, plot_confusion
from per_model import (compare_methods, draw_example_sets, enough_table, load_data, m_values_to_run, main_m_values,
                       main_results, plot_sensitivity, respect, run_draws, score_draws)

## Parameters

In [ ]:
LABELS_FILE = "human_labels.csv"
TEXT_COLUMN = "Transcript"
LABEL_COLUMN = "ability_label_name"
REFERENCE_COLUMN = None         # reference (instructed) level; None: a column whose name contains "ref" or "instruct"
GENERATOR_COLUMN = None         # LLM that generated each video; None: a column whose name contains "model", "generator" or "llm"
DIMENSION = "ability"           # names the outputs

M_VALUES = [1, 2, 3, 5, 7, 10, 15, 20]   # numbers of examples per level tested (too large values are skipped per model)
M_MAIN = 10                     # M of the main results: a number, or "auto" (smallest M "enough" per model)
N_DRAWS = 50                    # random draws of the examples, for each model, label and M
SEED = 0                        # same SEED in the three notebooks = same draws
N_BOOTSTRAP = 1000
TOLERANCE = 0.02                # "enough examples": within this of the best score over M
OUTPUT_DIR = "results"

EMBEDDING_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"
METRIC = "cosine"               # "cosine" or "euclidean"
CACHE_DIR = os.path.join(OUTPUT_DIR, "per_model_cache_embeddings")

## Labeled videos and generator models

In [ ]:
df, LABELS, GENERATORS, counts, max_m = load_data(LABELS_FILE, TEXT_COLUMN, LABEL_COLUMN, REFERENCE_COLUMN, GENERATOR_COLUMN)
M_RUN = m_values_to_run(M_VALUES, M_MAIN)
print("largest usable M per model and label:", max_m)
os.makedirs(OUTPUT_DIR, exist_ok=True)
counts

## Embeddings

In [ ]:
X = SentenceTransformer(EMBEDDING_MODEL).encode(df["text"].tolist(), batch_size=64)
if METRIC == "cosine":  # cosine distance = Euclidean distance between normalized vectors
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
vectors = dict(zip(df["id"], X))

## Draws

For each model, label and $M$: the mean vector of each level in each draw and the distances of the videos outside it (cached in `CACHE_DIR`).

In [ ]:
def classify_draw(generator, label, m, draw, chosen, targets):
    # Distances of the targets to the mean vector of each level's examples.
    centroids = np.array([np.mean([vectors[i] for i in chosen[level]], axis=0) for level in LEVELS])
    X_targets = np.array([vectors[i] for i in targets])
    distances = np.linalg.norm(X_targets[:, None, :] - centroids[None, :, :], axis=2)
    return pd.DataFrame({"id": targets, **{f"distance_{level}": distances[:, k] for k, level in enumerate(LEVELS)}})


draws = run_draws(df, LABELS, GENERATORS, max_m, M_RUN, N_DRAWS, SEED, classify_draw, CACHE_DIR, METRIC, DIMENSION)

## Scores with $M$ random examples

For each model, examples label, $M$ and label evaluated against: accuracy and quadratic-weighted $\kappa$ over all (draw, video) pairs, their 95% CI over draws, and the 5–95% range of the accuracy of a single draw. The `all` row pools the models (draw $d$ of each model together).

In [ ]:
scores = score_draws(draws, LABELS, GENERATORS, N_BOOTSTRAP, SEED)
scores.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_per_model_scores_{DIMENSION}.csv"), index=False)
scores.round(3)

## Sensitivity to the number of examples

Accuracy and $\kappa$ as a function of $M$ (examples per level), one line per model; the band is the 95% CI over draws. Rows: label of the examples; columns: label evaluated against.

In [ ]:
plot_sensitivity(scores, LABELS, GENERATORS, "accuracy", "accuracy", 1 / len(LEVELS),
                 os.path.join(OUTPUT_DIR, f"human_embedding_per_model_sensitivity_accuracy_{DIMENSION}.png"), "embeddings")
plot_sensitivity(scores, LABELS, GENERATORS, "kappa", "quadratic-weighted kappa", 0,
                 os.path.join(OUTPUT_DIR, f"human_embedding_per_model_sensitivity_kappa_{DIMENSION}.png"), "embeddings")

### How many examples are enough?

For each model, examples label and label evaluated against: the smallest $M$ whose score is within `TOLERANCE` of the best score over the tested $M$, and the spread of a single draw at that $M$. If the best score is reached at the largest $M$ available (`still rising`), more examples might still help.

In [ ]:
enough = enough_table(scores, TOLERANCE)
enough.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_per_model_enough_{DIMENSION}.csv"), index=False)
enough.round(3)

## Main results with `M_MAIN` examples

For each model, the scores with `M_MAIN` examples per level (the largest $M$ computed that does not exceed it; with `"auto"`, the $M$ "enough" for that model), and the confusion matrices over all (draw, video) pairs.

In [ ]:
main_keys = main_m_values(M_MAIN, draws, enough, GENERATORS, LABELS)
print("M used:", {f"{g}, {label} examples": m for (g, label), m in main_keys.items()})
main, main_scores = main_results(draws, scores, main_keys)
main_scores.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_per_model_main_{DIMENSION}.csv"), index=False)
main_scores.pivot_table(index=["examples", "evaluated against"], columns="generator",
                        values=["M", "accuracy [95% CI]", "kappa [95% CI]"], aggfunc="first")

In [ ]:
readable = main[main["pred"].isin(LEVELS)]
plot_confusion({(f"vs {target}", f"examples: {source}"): part.assign(true=part[column])
                for target, column in LABELS.items() for source, part in readable.groupby("examples")},
               "true", os.path.join(OUTPUT_DIR, f"human_embedding_per_model_confusion_{DIMENSION}.png"))
plt.show()

## Does each LLM respect the requested level?

For each model: the share of its videos whose level found by the method equals the reference level, with examples labeled by humans or by the reference (main $M$, over all draws; 95% CI over draws), and the share whose **human** level equals the reference level (95% CI over videos). $\chi^2$: equal shares across models, with one prediction per video (the level nearest on average over the draws, or the most frequent one).

In [ ]:
respect_table, respect_tests = respect(df, LABELS, GENERATORS, main, main_scores, N_BOOTSTRAP, SEED, "embeddings", DIMENSION,
                                       os.path.join(OUTPUT_DIR, f"human_embedding_per_model_respect_{DIMENSION}.png"), extra_modes=None)
if respect_table is not None:
    respect_table.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_per_model_respect_{DIMENSION}.csv"), index=False)

## Comparison of the methods

The sensitivity curves (all models pooled) and main scores of the per-model notebooks already run with the same parameters (`human_fgw_per_model`, `human_embeddings_per_model`, `human_llm_judge_per_model`).

In [ ]:
compare_methods(OUTPUT_DIR, DIMENSION, LABELS, "accuracy", os.path.join(OUTPUT_DIR, f"per_model_comparison_{DIMENSION}.png"))